<a href="https://colab.research.google.com/github/Parul-gargg/Project-2-FitZone/blob/main/FitZone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏋️ FitZone Gym — Customer Churn & Retention Analysis

**FitZone** is a single gym in Gurugram: 1,200 members joined over 18 months (Jan 2025 – Jun 2026) on three plans — Monthly (₹1,500/mo), Quarterly (₹1,200/mo), Annual (₹1,000/mo). The owner, Kavita, wants answers:






## Setup — Import Libraries & Load the Raw Data


In [ ]:
import os

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
members = pd.read_csv('members.csv')
visits = pd.read_csv('visits.csv')


In [ ]:
print(members.shape)

(1200, 8)


In [ ]:
visits.shape

(80351, 3)

In [ ]:
members.head()

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,member_id,join_date,age,plan,monthly_fee_inr,signup_source,status,cancel_date
0,M2000,31-07-2025,25,Monthly,1500,Walk-in,Active,NaN
1,M2001,12-04-2026,29,Quarterly,1200,Instagram,Active,NaN
2,M2002,29-05-2025,25,Quarterly,1200,Instagram,Active,NaN
3,M2003,14-01-2026,34,Annual,1000,Instagram,Active,NaN
4,M2004,23-02-2026,34,Monthly,1500,Instagram,Cancelled,21-03-2026


In [ ]:
visits.head()

,member_id,visit_date,checkin_hour
0,M2769,16-05-2026,8
1,M2048,13-04-2026,21
2,M2537,28-11-2025,6
3,M2523,29-12-2025,9
4,M3078,16-04-2026,9


In [ ]:
members.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   member_id        1200 non-null   object
 1   join_date        1200 non-null   object
 2   age              1200 non-null   int64 
 3   plan             1200 non-null   object
 4   monthly_fee_inr  1200 non-null   int64 
 5   signup_source    1200 non-null   object
 6   status           1200 non-null   object
 7   cancel_date      481 non-null    object
dtypes: int64(2), object(6)
memory usage: 75.1+ KB


In [ ]:
visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80351 entries, 0 to 80350
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   member_id     80351 non-null  object
 1   visit_date    80351 non-null  object
 2   checkin_hour  80351 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.8+ MB


## Task 1 — Clean the Data




### Check for missing values in both tables


In [ ]:
members.isnull().sum()

,0
member_id,0
join_date,0
age,0
plan,0
monthly_fee_inr,0
signup_source,0
status,0
cancel_date,719


In [ ]:
visits.isnull().sum()

,0
member_id,0
visit_date,0
checkin_hour,0


In [ ]:
members.groupby("status")["cancel_date"].apply(lambda x: x.isna().sum())

,cancel_date
status,
Active,719
Cancelled,0


### Check for duplicate members and duplicate visit check-ins


In [ ]:
members["member_id"].duplicated().sum()

np.int64(0)

In [ ]:
visits.duplicated().sum()

np.int64(2555)

In [ ]:
visits[visits.duplicated()].head(10)

,member_id,visit_date,checkin_hour
2422,M2122,20-06-2026,9
2683,M2333,30-05-2026,9
3034,M2883,03-06-2026,20
3113,M2389,20-01-2026,8
3206,M2128,10-05-2025,19
3331,M2051,02-09-2025,18
4147,M2799,24-04-2026,18
4498,M2896,28-10-2025,19
4967,M2915,27-08-2025,20
4972,M2227,23-10-2025,20


In [ ]:
visits[visits.duplicated()].shape

(2555, 3)

### Remove duplicate check-in rows from `visits`


In [ ]:
visits_clean = visits.drop_duplicates()

In [ ]:
visits_clean.shape

(77796, 3)

In [ ]:
visits_clean.duplicated().sum()

np.int64(0)

In [ ]:
visits_clean["member_id"].isin(members["member_id"]).value_counts()

,count
member_id,
True,77796


### Parse `join_date` / `cancel_date`, and flag impossible cancel-before-join rows

Rather than deleting these members outright, flag them with a `date_error` column so the corrupt field can be handled without throwing away a real member's other data.


In [ ]:
members["join_date"] = pd.to_datetime(members["join_date"], dayfirst=True)
members["cancel_date"] = pd.to_datetime(members["cancel_date"], dayfirst=True)

In [ ]:
members[members["cancel_date"] < members["join_date"]]

,member_id,join_date,age,plan,monthly_fee_inr,signup_source,status,cancel_date
117,M2117,2025-01-17,36,Monthly,1500,Walk-in,Cancelled,2024-12-18
173,M2173,2026-01-25,27,Monthly,1500,Referral,Cancelled,2025-12-26
452,M2452,2025-01-20,17,Monthly,1500,Instagram,Cancelled,2024-12-21
857,M2857,2026-04-29,42,Quarterly,1200,Walk-in,Cancelled,2026-03-30
970,M2970,2026-01-29,26,Annual,1000,Google,Cancelled,2025-12-30
1099,M3099,2026-02-14,21,Quarterly,1200,Instagram,Cancelled,2026-01-15


In [ ]:
members["date_error"] = members["cancel_date"] < members["join_date"]

In [ ]:
members["date_error"].value_counts()

,count
date_error,
False,1194
True,6


In [ ]:
visits_clean = visits.drop_duplicates().copy()

In [ ]:
visits_clean["visit_date"] = pd.to_datetime(
    visits_clean["visit_date"], dayfirst = True
)

In [ ]:
visits_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 77796 entries, 0 to 80350
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   member_id     77796 non-null  object        
 1   visit_date    77796 non-null  datetime64[ns]
 2   checkin_hour  77796 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 2.4+ MB


### Sanity check — any gym visits logged after a member's cancel date?

Merge each visit with the member's `cancel_date` and check for check-ins that happened after cancellation, excluding the rows already flagged as date errors.


In [ ]:
visits_check = visits_clean.merge(
    members[["member_id", "cancel_date"]],
    on = "member_id",
    how = "left"
)

In [ ]:
post_cancel = visits_check[
    visits_check ["visit_date"] > visits_check["cancel_date"]]

In [ ]:
post_cancel.shape

(94, 4)

In [ ]:
post_cancel.head(10)

,member_id,visit_date,checkin_hour,cancel_date
8,M2452,2025-04-10,20,2024-12-21
1400,M2452,2025-04-07,19,2024-12-21
2218,M2117,2025-12-22,8,2024-12-18
2276,M2452,2025-04-15,17,2024-12-21
2483,M2117,2026-04-09,17,2024-12-18
2632,M2857,2026-06-02,18,2026-03-30
2683,M2117,2026-05-24,19,2024-12-18
3331,M2117,2025-10-03,17,2024-12-18
5674,M2117,2026-04-07,9,2024-12-18
6830,M2117,2025-10-19,8,2024-12-18


In [ ]:
post_cancel_valid = post_cancel[
    post_cancel["member_id"].isin(
        members.loc[members["date_error"], "member_id"]
    ) == False
]

In [ ]:
post_cancel_valid.shape

(0, 4)

## Task 2 — Overall Churn (and why it's almost useless on its own)

Compute the headline churn rate across all 1,200 members. On its own this number has no benchmark to compare against — the real value comes from splitting it by plan, source, and behaviour in the tasks below.


In [ ]:
members["status"].value_counts()

,count
status,
Active,719
Cancelled,481


In [ ]:
churn_rate = (
    (members["status"] == "Cancelled").sum()
    / len(members)
) * 100

churn_rate


np.float64(40.08333333333333)

## Task 3 — Churn by Plan

Break the churn rate down by `plan` (Monthly / Quarterly / Annual) to see whether commitment length is linked to how likely a member is to leave.


In [ ]:
pd.crosstab(members["plan"], members["status"])

status,Active,Cancelled
plan,,
Annual,216,36
Monthly,251,323
Quarterly,252,122


In [ ]:
plan_churn = pd.crosstab(
    members["plan"],
    members["status"],
    normalize = "index"
)*100

plan_churn = plan_churn.round(2)

plan_churn

status,Active,Cancelled
plan,,
Annual,85.71,14.29
Monthly,43.73,56.27
Quarterly,67.38,32.62


## Task 4 — Churn by Signup Source

Break the churn rate down by `signup_source` (Instagram, Google, Walk-in, Referral) to see which acquisition channel brings in members who actually stick around.


In [ ]:
pd.crosstab(
    members["signup_source"],
    members["status"]
)

status,Active,Cancelled
signup_source,,
Google,125,79
Instagram,202,192
Referral,162,65
Walk-in,230,145


In [ ]:
source_churn = pd.crosstab(
    members["signup_source"],
    members["status"],
    normalize = "index"
)*100

In [ ]:
source_churn.round(2)

status,Active,Cancelled
signup_source,,
Google,61.27,38.73
Instagram,51.27,48.73
Referral,71.37,28.63
Walk-in,61.33,38.67


## Task 5 — The January Effect

Members who join in January often join on New Year's resolution motivation rather than an established habit. Break churn down by `join_month` to check whether January joiners churn at a higher rate than everyone else.


In [ ]:
members["join_month"] = members["join_date"].dt.month

In [ ]:
members["join_month"]

,join_month
0,7
1,4
2,5
3,1
4,2
...,...
1195,10
1196,10
1197,4
1198,1


In [ ]:
members["join_month"].value_counts().sort_index()

,count
join_month,
1,409
2,103
3,124
4,86
5,49
6,57
7,59
8,64
9,58


In [ ]:
monthly_churn = pd.crosstab(
    members["join_month"],
    members["status"],
    normalize="index"
) * 100

In [ ]:
monthly_churn.round(2)

status,Active,Cancelled
join_month,,
1,52.81,47.19
2,51.46,48.54
3,62.10,37.90
4,69.77,30.23
5,61.22,38.78
6,59.65,40.35
7,64.41,35.59
8,67.19,32.81
9,67.24,32.76


## Task 6 — The Habit Finding (the star of this project)

**The idea:** churn correlates with what people *do*, not just what they *bought*. Count each member's visits in their **first 30 days**, bucket the counts, and compare churn across buckets. Unlike plan or source, this is a signal Kavita can act on in real time — before the member is gone.


In [ ]:
visits_habit = visits_clean.merge(
    members[["member_id", "join_date", "status"]],
    on = "member_id",
    how = "left"

)

In [ ]:
visits_habit

,member_id,visit_date,checkin_hour,join_date,status
0,M2769,2026-05-16,8,2026-01-22,Active
1,M2048,2026-04-13,21,2025-02-17,Active
2,M2537,2025-11-28,6,2025-01-03,Active
3,M2523,2025-12-29,9,2025-06-27,Active
4,M3078,2026-04-16,9,2026-01-28,Cancelled
...,...,...,...,...,...
77791,M3100,2026-02-13,20,2025-02-04,Active
77792,M2978,2026-04-10,18,2025-05-20,Active
77793,M2380,2026-03-09,10,2025-04-18,Active
77794,M2179,2026-06-24,9,2026-01-03,Active


In [ ]:
visits_habit["days_since_join"] =  (visits_habit["visit_date"] - visits_habit["join_date"]
).dt.days

In [ ]:
visits_habit

,member_id,visit_date,checkin_hour,join_date,status,days_since_join
0,M2769,2026-05-16,8,2026-01-22,Active,114
1,M2048,2026-04-13,21,2025-02-17,Active,420
2,M2537,2025-11-28,6,2025-01-03,Active,329
3,M2523,2025-12-29,9,2025-06-27,Active,185
4,M3078,2026-04-16,9,2026-01-28,Cancelled,78
...,...,...,...,...,...,...
77791,M3100,2026-02-13,20,2025-02-04,Active,374
77792,M2978,2026-04-10,18,2025-05-20,Active,325
77793,M2380,2026-03-09,10,2025-04-18,Active,325
77794,M2179,2026-06-24,9,2026-01-03,Active,172


In [ ]:
visits_habit[["member_id", "visit_date", "join_date", "days_since_join"]].head()

,member_id,visit_date,join_date,days_since_join
0,M2769,2026-05-16,2026-01-22,114
1,M2048,2026-04-13,2025-02-17,420
2,M2537,2025-11-28,2025-01-03,329
3,M2523,2025-12-29,2025-06-27,185
4,M3078,2026-04-16,2026-01-28,78


### Keep only visits within the first 30 days of joining

Attach each member's `join_date` to every visit, compute `days_since_join`, and filter down to the first 30-day window.


In [ ]:
first30 = visits_habit[visits_habit["days_since_join"].between(0,30)]

In [ ]:
first30

,member_id,visit_date,checkin_hour,join_date,status,days_since_join
13,M3151,2026-01-19,19,2026-01-12,Active,7
22,M2007,2025-08-04,20,2025-07-15,Cancelled,20
27,M2736,2025-10-13,19,2025-10-06,Active,7
33,M2869,2026-04-15,9,2026-03-31,Active,15
36,M2121,2026-03-24,17,2026-03-09,Active,15
...,...,...,...,...,...,...
77774,M3024,2025-05-19,18,2025-05-05,Cancelled,14
77777,M3197,2025-04-26,20,2025-04-01,Active,25
77782,M2857,2026-05-09,6,2026-04-29,Cancelled,10
77784,M2417,2025-02-04,20,2025-01-13,Active,22


In [ ]:
first30.shape

(10198, 6)

### Count each member's visit total in their first 30 days


In [ ]:
first30_visits = first30.groupby("member_id").size()

In [ ]:
first30_visits.head()

,0
member_id,
M2000,12
M2001,6
M2002,6
M2003,5
M2004,13


In [ ]:
member_habit = members[["member_id", "status"]].copy()


In [ ]:
member_habit["first30_visits"] =(
    member_habit["member_id"].map(first30_visits).fillna(0).astype(int)
)

In [ ]:
member_habit.head(10)

,member_id,status,first30_visits
0,M2000,Active,12
1,M2001,Active,6
2,M2002,Active,6
3,M2003,Active,5
4,M2004,Cancelled,13
5,M2005,Cancelled,7
6,M2006,Active,13
7,M2007,Cancelled,13
8,M2008,Active,15
9,M2009,Cancelled,6


### Bucket members by early-visit count

Members with zero early visits must still be included (`fillna(0)`) — they're exactly the highest-risk group, and dropping them would wreck the finding.


In [ ]:
member_habit["visit_bucket"] = pd.cut(
    member_habit["first30_visits"],
    bins=[-1, 2, 5, 10, float("inf")],
    labels=["0-2", "3-5", "6-10", "11+"]
)

In [ ]:
member_habit.head(10)

,member_id,status,first30_visits,visit_bucket
0,M2000,Active,12,11+
1,M2001,Active,6,6-10
2,M2002,Active,6,6-10
3,M2003,Active,5,3-5
4,M2004,Cancelled,13,11+
5,M2005,Cancelled,7,6-10
6,M2006,Active,13,11+
7,M2007,Cancelled,13,11+
8,M2008,Active,15,11+
9,M2009,Cancelled,6,6-10


### Compare churn rate across visit buckets


In [ ]:
habit_churn = pd.crosstab(
    member_habit["visit_bucket"],
    member_habit["status"],
    normalize="index"
) * 100

In [ ]:
habit_churn.round(2)

status,Active,Cancelled
visit_bucket,,
0-2,41.82,58.18
3-5,44.31,55.69
6-10,58.87,41.13
11+,74.53,25.47


## Task 7 — The Memo to Kavita *(to do)*

Turn the findings above into concrete, numbered recommendations, each backed by a number from this notebook:

1. **Day-20 rescue list** — flag members with low early-visit counts (Task 6) for a call/WhatsApp + free PT session before they churn.
2. **Push commitment at signup** — Monthly plans churn far more than Annual (Task 3); a signup discount on longer plans costs less than the churn it prevents.
3. **Shift acquisition budget** — compare Instagram vs Referral churn (Task 4), and auto-enrol January joiners (Task 5) into the Day-20 rescue process.

⚠️ **Caveat to state explicitly:** low early visits *correlating* with churn doesn't prove visits *cause* retention — but as an early-warning flag, the signal is useful either way, no causal claim required.
